In [ ]:
import pandas as pd
import os

BASE_DIR = "C:/Tesis"


## 1) Filtrado store_nbr = 44

In [ ]:
FILE_PATH = os.path.join(BASE_DIR, "union_bases_sin_filtros.csv")
CHUNK_SIZE = 1_000_000
STORE_NBR = 44

tamano_gb = os.path.getsize(FILE_PATH) / (1024**3)
print(f"Tamaño en disco: {tamano_gb:.2f} GB")


In [ ]:
partes = []

for i, chunk in enumerate(pd.read_csv(FILE_PATH, chunksize=CHUNK_SIZE, parse_dates=["date"])):
    partes.append(chunk[chunk["store_nbr"] == STORE_NBR])

    if (i + 1) % 5 == 0:
        print(f"Procesadas {(i + 1) * CHUNK_SIZE:,} filas leídas del archivo...")

bbdd_filtrada = pd.concat(partes, ignore_index=True)
print(f"\nFilas de la tienda 44: {len(bbdd_filtrada):,}")


## Revisión rápida

In [ ]:
bbdd_filtrada.info()


In [ ]:
bbdd_filtrada.head(10)


## 2) Filtro de feriados (is_holiday)

In [ ]:
# --- Filtro de holidays_events ---
# Rellenar nulos: fechas sin fila de holidays_events = sin feriado/evento ese dia
cols_holiday_texto = ["holiday_type", "locale", "locale_name", "description"]
for col in cols_holiday_texto:
    bbdd_filtrada[col] = bbdd_filtrada[col].fillna("no_feriado")

bbdd_filtrada["transferred"] = bbdd_filtrada["transferred"].fillna(False).astype(bool)

# Locale relevante para Store 44: nacional, o local/regional de Quito/Pichincha
mask_locale = (
    (bbdd_filtrada["locale"] == "National") |
    (bbdd_filtrada["locale_name"].str.lower().isin(["quito", "pichincha"]))
)

# feriado real: Holiday no trasladado, o Transfer/Additional/Bridge
# (excluye implicitamente Work Day y Holiday con transferred=True)
bbdd_filtrada["feriado"] = mask_locale & (
    ((bbdd_filtrada["holiday_type"] == "Holiday") & (~bbdd_filtrada["transferred"])) |
    (bbdd_filtrada["holiday_type"].isin(["Transfer", "Additional", "Bridge"]))
)

# evento: no es dia no laboral, pero puede mover la demanda
bbdd_filtrada["evento"] = mask_locale & (bbdd_filtrada["holiday_type"] == "Event")

print(bbdd_filtrada["feriado"].value_counts())
print(bbdd_filtrada["evento"].value_counts())


## 3) Relleno de precio de petróleo

In [ ]:
# --- Relleno de dcoilwtico (precio del petróleo) ---
# El petróleo no cotiza fines de semana/feriados, por eso hay NaN en esas fechas.
# Se ordena por fecha y se aplica forward-fill (arrastra el último valor conocido);
# se agrega un backward-fill solo por si la serie empieza con NaN (sin dato previo).
bbdd_filtrada = bbdd_filtrada.sort_values("date").reset_index(drop=True)

n_nulos_antes = bbdd_filtrada["dcoilwtico"].isna().sum()

bbdd_filtrada["dcoilwtico"] = bbdd_filtrada["dcoilwtico"].ffill()
bbdd_filtrada["dcoilwtico"] = bbdd_filtrada["dcoilwtico"].bfill()

n_nulos_despues = bbdd_filtrada["dcoilwtico"].isna().sum()
print(f"Nulos en dcoilwtico antes: {n_nulos_antes:,}")
print(f"Nulos en dcoilwtico después: {n_nulos_despues:,}")


In [ ]:
# Revisión rápida del resultado
bbdd_filtrada[["date", "holiday_type", "locale", "locale_name", "feriado", "dcoilwtico"]].head(10)


## Estadística descriptiva

### Nulos y duplicados

In [ ]:
# Nulos por columna
nulos = bbdd_filtrada.isna().sum()
nulos = nulos[nulos > 0].sort_values(ascending=False)
print("Columnas con valores nulos:")
print(nulos)


In [ ]:
# Duplicados por 'id' (cada fila debería ser única)
n_duplicados_id = bbdd_filtrada["id"].duplicated().sum()
print(f"Filas con 'id' duplicado: {n_duplicados_id:,}")

# Duplicados considerando toda la fila
n_duplicados_totales = bbdd_filtrada.duplicated().sum()
print(f"Filas completamente duplicadas: {n_duplicados_totales:,}")


### Estadísticas descriptivas — variables numéricas

In [ ]:
# describe() sobre todas las columnas numéricas
cols_numericas = bbdd_filtrada.select_dtypes(include="number").columns.tolist()
print("Columnas numéricas:", cols_numericas)

bbdd_filtrada[cols_numericas].describe().T.round(2)


### Estadísticas descriptivas — variables categóricas / booleanas

In [ ]:
# describe() sobre columnas categóricas (incluye texto y booleanas)
cols_categoricas = bbdd_filtrada.select_dtypes(include=["object", "bool"]).columns.tolist()
print("Columnas categóricas:", cols_categoricas)

bbdd_filtrada[cols_categoricas].describe().T


In [ ]:
# Detalle de frecuencias para las categóricas más relevantes
cols_categoricas_clave = [
    "family", "store_type", "city", "state",
    "holiday_type", "locale", "locale_name",
    "is_holiday", "onpromotion", "transferred"
]

for col in cols_categoricas_clave:
    if col in bbdd_filtrada.columns:
        print(f"\n--- {col} ---")
        print(bbdd_filtrada[col].value_counts(dropna=False))


### `unit_sales` según `is_holiday` y `onpromotion`

In [ ]:
# Estadísticas de unit_sales agrupadas por is_holiday
resumen_holiday = bbdd_filtrada.groupby("is_holiday")["unit_sales"].agg(
    n="count", media="mean", mediana="median", desv_std="std", minimo="min", maximo="max"
)
print("unit_sales por is_holiday:")
resumen_holiday


In [ ]:
# Estadísticas de unit_sales agrupadas por onpromotion
resumen_promo = bbdd_filtrada.groupby("onpromotion")["unit_sales"].agg(
    n="count", media="mean", mediana="median", desv_std="std", minimo="min", maximo="max"
)
print("unit_sales por onpromotion:")
resumen_promo


In [ ]:
# Cruce is_holiday x onpromotion: media de unit_sales en cada combinación
resumen_cruzado = bbdd_filtrada.groupby(["is_holiday", "onpromotion"])["unit_sales"].agg(
    n="count", media="mean", mediana="median"
)
print("unit_sales por is_holiday x onpromotion:")
resumen_cruzado


### Distribución de categorías (`family`)

In [ ]:
# Conteo y porcentaje de cada categoría (family)
distribucion_family = bbdd_filtrada["Category"].value_counts().reset_index()
distribucion_family.columns = ["Category", "n"]
distribucion_family["porcentaje"] = (distribucion_family["n"] / len(bbdd_filtrada) * 100).round(2)

print(f"Total de categorías distintas: {bbdd_filtrada['Category'].nunique()}")
distribucion_family


In [ ]:
import matplotlib.pyplot as plt

# Gráfico de barras con las categorías más frecuentes
TOP_N = 15
top_categorias = distribucion_family.head(TOP_N)

plt.figure(figsize=(10, 6))
plt.barh(top_categorias["Category"][::-1], top_categorias["n"][::-1])
plt.xlabel("Cantidad de registros")
plt.title(f"Top {TOP_N} categorías (Category) — Tienda 44")
plt.tight_layout()
plt.show()
